# Installation

In [ ]:
#pip uninstall -y transformers torch torchvision

In [ ]:
#!pip install git+https://github.com/dnth/rag-datakit.git

In [1]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

In [2]:
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10")
dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1125
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 282
    })
})

In [3]:
dataset['valid'][0]

{'anchor': 'The Engineering Head (Power) is a subject matter expert on rail power systems. He/She leads the organisation to implement rail power systems maintenance regime and improvement strategies. His duties also include translating and aligning established industry standards into department Key Performance Indicators (KPIs). He possesses a strong understanding of leading engineering practices, operational best practices, industry developments and regulatory requirements and he translates these into organisation practices and performance requirements. He possesses strong leadership skills, is able to cultivate a culture of continuous improvement and demonstrates excellent management skills to achieve the departments operational and functional goals.',
 'positive': 'The Engineering Head (Power) serves as a subject matter expert on rail power systems, guiding the organization in implementing maintenance regimes and strategies for improvement. His responsibilities also involve translat

# W&B and Model Configuration

In [4]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Fetch the WANDB_API_KEY from the environment
wandb_api_key = os.getenv("WAB_API_KEY")

# Log in using the API key
wandb.login(key=wandb_api_key)

model_id = "Qwen/Qwen3-Embedding-0.6B"
save_model_path = "./models/Qwen/Qwen3-Embedding-0.6B"

wandb.init(project="rag-datakit-finetunes", name="Attempt_4_Qwen3-Embedding-0.6B~frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: frankwong2001 (frankwong2001-cxsanalytics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Training Arguments

In [5]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",
    tf32=False,                                 # use tf32 precision
    bf16=True,         
    #fp16=True,                                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb",
    gradient_checkpointing=True,              # use fused adamw optimizer
    #use_cache=False                            # disable the use of cache
    weight_decay=0.01,                             # apply weight decay
    max_grad_norm=0.5,                           # clip the gradient norm
    warmup_steps=1500                           # number of warmup steps
    )

In [6]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# Execute Training


In [7]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.013900,0.018476
2,0.017800,0.018441
3,0.014400,0.018429
4,0.013500,0.018424
5,0.015000,0.018080


TrainOutput(global_step=15, training_loss=0.014926230907440186, metrics={'train_runtime': 341.4786, 'train_samples_per_second': 16.472, 'train_steps_per_second': 0.044, 'total_flos': 0.0, 'train_loss': 0.014926230907440186, 'epoch': 5.0})

#  Save & Upload Model

In [8]:
trainer.save_model()

In [9]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 19 files into the W&B run directory, call wandb.save again to sync new files.


['/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/sentence_bert_config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/added_tokens.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/chat_template.jinja',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/checkpoint-9',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/merges.txt',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/model.safetensors',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250908_093941-cwkehna3/files/models/Qwen/Qwen3-Embedding-0.6B/2_Normalize',
 '/root/rag-datakit/nb

In [10]:
wandb.finish()

eval/loss,█▇▇▇▁
eval/runtime,▁▅▆█▇
eval/samples_per_second,█▄▃▁▂
eval/steps_per_second,█▄▃▁▂
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,▇█▅▁▂
train/learning_rate,▁▃▅▆█
train/loss,▂█▂▁▃
eval/loss,0.01808
eval/runtime,2.7622


# Push to Hugging Face

In [11]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import Trainer

# Load environment variables from .env file
load_dotenv()

# Fetch the Hugging Face API key from the environment
hf_api_key = os.getenv("HF_TOKEN")

# Log in using the Hugging Face API key
login(token=hf_api_key)

# Assuming you have a Trainer object `trainer`
trainer.model.push_to_hub("frankwong2001/4_attempt_Qwen3-Embedding-0.6B", exist_ok=True)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpsyysw9xx/tokenizer.json       :   0%|          | 29.7kB / 11.4MB            

  /tmp/tmpsyysw9xx/model.safetensors    :   0%|          |  957kB / 2.38GB            

'https://huggingface.co/frankwong2001/4_attempt_Qwen3-Embedding-0.6B/commit/15f85474e56bb2ff5d627b1dd78f69143967bd90'